# pandas 基礎 II — groupby・結合・pivot・apply

groupby による集計、merge/join での結合、pivot_table、apply/map による変換を学びます。

> **Kaggle Tutorial** 対応ノートブック（`/learn/python-data-analysis/04-pandas-aggregation`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# [Colab / ローカル環境用セットアップ]
# Kaggle Notebook 上では既存の /kaggle/input/titanic をそのまま使用し、
# Google Colab やローカル環境では互換スキーマの Titanic サンプルデータ（891行 / 418行）を自動配置します。
# ※ Kaggle 本番データで提出する場合は kaggle CLI (!kaggle competitions download -c titanic) をご利用ください。
import os
import numpy as np
import pandas as pd

def _ensure_titanic_data():
    target_dir = "/kaggle/input/titanic"
    if os.path.exists(os.path.join(target_dir, "train.csv")):
        return

    try:
        os.makedirs(target_dir, exist_ok=True)
        data_dir = target_dir
    except OSError:
        data_dir = os.path.abspath("./.kaggle_input/titanic")
        os.makedirs(data_dir, exist_ok=True)
        _orig_read_csv = pd.read_csv

        def _compat_read_csv(filepath_or_buffer, *args, **kwargs):
            if isinstance(filepath_or_buffer, str) and filepath_or_buffer.startswith("/kaggle/input/"):
                rel = filepath_or_buffer[len("/kaggle/input/"):]
                local_path = os.path.abspath(os.path.join("./.kaggle_input", rel))
                if os.path.exists(local_path):
                    filepath_or_buffer = local_path
            return _orig_read_csv(filepath_or_buffer, *args, **kwargs)

        pd.read_csv = _compat_read_csv

    rng = np.random.default_rng(42)
    surnames = ["Braund", "Cumings", "Heikkinen", "Futrelle", "Allen", "Moran", "McCarthy", "Palsson", "Johnson", "Nasser"]
    given_m = ["Owen", "William", "James", "Henry", "Leonard", "Timothy", "charles", "George"]
    given_f = ["Florence", "Laina", "Lily", "elisabeth", "Adele", " Marguerite", "Anna", "Mary"]

    def _make_split(n, start_id, with_target):
        pclass = rng.choice([1, 2, 3], size=n, p=[0.24, 0.21, 0.55])
        sex = rng.choice(["male", "female"], size=n, p=[0.65, 0.35])
        age = np.clip(rng.normal(29.7, 14.5, size=n), 0.42, 80.0).round(1)
        sibsp = rng.choice([0, 1, 2, 3, 4], size=n, p=[0.68, 0.23, 0.05, 0.02, 0.02])
        parch = rng.choice([0, 1, 2, 3], size=n, p=[0.76, 0.13, 0.09, 0.02])
        base_fare = np.where(pclass == 1, 60.0, np.where(pclass == 2, 20.0, 11.0))
        fare = (base_fare * rng.lognormal(0.0, 0.55, size=n)).round(2)
        embarked = rng.choice(["S", "C", "Q"], size=n, p=[0.72, 0.19, 0.09]).astype(object)

        names = []
        cabins = []
        tickets = []
        for i in range(n):
            sn = surnames[i % len(surnames)]
            if sex[i] == "male":
                title = "Master" if age[i] < 13 else "Mr"
                gn = given_m[i % len(given_m)].strip().capitalize()
            else:
                title = "Miss" if age[i] < 24 else "Mrs"
                gn = given_f[i % len(given_f)].strip().capitalize()
            names.append(f"{sn}, {title}. {gn}")
            cabins.append(f"{'ABC'[pclass[i] - 1]}{10 + (i % 90)}" if rng.random() > 0.77 else np.nan)
            tickets.append(f"PC {17500 + (i % 350)}" if pclass[i] == 1 else str(230000 + i))

        age_obj = age.astype(float)
        age_obj[rng.random(n) < 0.20] = np.nan
        embarked[rng.random(n) < 0.005] = np.nan
        if not with_target:
            fare[0] = np.nan

        data = {
            "PassengerId": np.arange(start_id, start_id + n),
            "Pclass": pclass,
            "Name": names,
            "Sex": sex,
            "Age": age_obj,
            "SibSp": sibsp,
            "Parch": parch,
            "Ticket": tickets,
            "Fare": fare,
            "Cabin": pd.Series(cabins, dtype=object),
            "Embarked": pd.Series(embarked, dtype=object),
        }
        if with_target:
            family_size = sibsp + parch + 1
            logit = (
                -1.65
                + np.where(sex == "female", 2.45, 0.0)
                + np.where(pclass == 1, 1.35, np.where(pclass == 2, 0.35, -0.55))
                + np.where(np.nan_to_num(age_obj, nan=29.0) < 12, 1.2, 0.0)
                + np.where((family_size >= 2) & (family_size <= 4), 0.75, -0.35)
            )
            prob = 1.0 / (1.0 + np.exp(-logit))
            survived = (rng.random(n) < prob).astype(int)
            ordered = {"PassengerId": data["PassengerId"], "Survived": survived}
            for k, v in data.items():
                if k != "PassengerId":
                    ordered[k] = v
            return pd.DataFrame(ordered)
        return pd.DataFrame(data)

    _make_split(891, 1, True).to_csv(os.path.join(data_dir, "train.csv"), index=False)
    _make_split(418, 892, False).to_csv(os.path.join(data_dir, "test.csv"), index=False)
    print(f"Titanic dataset ready at: {data_dir}")

_ensure_titanic_data()

## groupby：カテゴリごとの集計

`groupby` は「分割 → 集計 → 結合（split-apply-combine）」を一手で行います。
カテゴリごとの傾向を見る EDA の主役です。

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/titanic/train.csv")

# 客室クラスごとの生存率
df.groupby("Pclass")["Survived"].mean()

# 複数キー・複数集計
df.groupby(["Pclass", "Sex"]).agg(
    survival_rate=("Survived", "mean"),
    count=("Survived", "size"),
)

`agg` に「新しい列名=(対象列, 集計関数)」を渡すと、結果が読みやすくなります。

## merge / join：表の結合

複数のテーブルを共通キーで結合します。SQL の JOIN に相当します。

In [ ]:
# 乗客テーブルに、客室クラスごとの平均運賃を結合
fare_by_class = (
    df.groupby("Pclass")["Fare"].mean().rename("fare_mean").reset_index()
)
merged = df.merge(fare_by_class, on="Pclass", how="left")

`how` で結合方法を選びます。欠損を増やしたくない特徴量結合では `how="left"` が基本です。

> **Warning:** 結合キーに重複があると行数が意図せず増えます（行の水増し）。 `merge` の前後で `shape` を確認し、想定どおりの行数かをチェックしましょう。

## pivot_table：クロス集計

行・列の 2 軸でクロス集計します。Excel のピボットテーブルと同じ発想です。

In [ ]:
df.pivot_table(
    index="Pclass", columns="Sex", values="Survived", aggfunc="mean"
)

## apply / map：要素・行ごとの変換

ベクトル化で書けない複雑な変換は `apply`、値の置換は `map` を使います。

In [ ]:
# 敬称（Title）を名前から抽出
df["Title"] = df["Name"].apply(lambda s: s.split(",")[1].split(".")[0].strip())

# カテゴリを数値へ写像
df["Sex_code"] = df["Sex"].map({"male": 0, "female": 1})

In [ ]:
df[["Name", "Title", "Sex", "Sex_code"]].head()

> **Tip:** `apply` は行ごとに Python 関数を呼ぶため低速です。`map` の辞書置換や、 `df["a"] + df["b"]` のようなベクトル化で書けないかを先に検討しましょう。

## 次のステップ

次は matplotlib / seaborn を使ったデータ可視化に進みます。